# arXiv Radar Notebook Workflow

This notebook is the recommended interactive way to use the `arxiv-radar` codebase.

Use `uv` only to manage the package environment:

```bash
cd /Users/wangyangming/research_project/arxiv-radar
uv sync
```

Then open this notebook and select the project interpreter as the kernel:

```text
/Users/wangyangming/research_project/arxiv-radar/.venv/bin/python
```

The cells below interact with the Python package directly. They do not shell out to `uv run arxiv-radar`.

## Setup

Run this first. It makes the notebook robust whether it is opened from the project root or from the `notebooks/` directory.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

PosixPath('/Users/wangyangming/research_project/arxiv-radar')

## Inspect the research profile

The YAML file is the persistent place to tune categories, keywords, and the optional author watchlist.

In [2]:
from arxiv_radar.config import load_config, write_default_config

CONFIG_PATH = PROJECT_ROOT / "arxiv-radar.yaml"
if not CONFIG_PATH.exists():
    write_default_config(CONFIG_PATH)

config = load_config(CONFIG_PATH)
config.model_dump()

{'categories': ['quant-ph', 'physics.optics', 'cond-mat.quant-gas', 'math-ph'],
 'high_priority_keywords': ['waveguide QED',
  'few-photon scattering',
  'multi-photon scattering',
  'photon scattering',
  'atomic arrays',
  'subwavelength arrays'],
 'method_keywords': ['Yudson',
  'Bethe ansatz',
  'Lippmann-Schwinger',
  'resolvent',
  'cumulant expansion'],
 'broader_discovery_keywords': ['topological quantum optics',
  'quantum nonlinear optics',
  'chiral quantum optics',
  'collective decay',
  'dark states',
  'spin waves'],
 'author_watchlist': ['Susanne Yelin',
  'Helmut Ritsch',
  'Michael Gullans',
  'Darrick E. Chang']}

In [3]:
print("Categories")
for category in config.categories:
    print("-", category)

print("\nHigh-priority keywords")
for keyword in config.high_priority_keywords:
    print("-", keyword)

Categories
- quant-ph
- physics.optics
- cond-mat.quant-gas
- math-ph

High-priority keywords
- waveguide QED
- few-photon scattering
- multi-photon scattering
- photon scattering
- atomic arrays
- subwavelength arrays


## Run arXiv Radar from Python

`run_radar` is the notebook-facing workflow function. It performs the same pipeline as the CLI:

1. load config,
2. query the official arXiv API,
3. filter recent papers with matched keywords,
4. score and sort papers,
5. return a notebook-friendly pandas table.

Keep `max_results` modest while exploring. The API client respects arXiv politeness when multiple pages are requested.

In [4]:
from arxiv_radar.workflow import run_radar

radar = run_radar(
    config_path=CONFIG_PATH,
    days=14,
    max_results=50,
)

len(radar.raw_papers), len(radar.papers)

(50, 6)

## Work with the ranked paper table

`radar.dataframe` is usually the easiest object to inspect interactively.

In [5]:
radar.dataframe.head(10)

,score,published,updated,title,authors,matched_keywords,primary_category,categories,arxiv_id,abstract_url,pdf_url,summary
0,13,2026-06-25T10:05:27+00:00,2026-06-25T10:05:27+00:00,Quantitative uniform resolvent estimates,Piero D'Ancona; Jérémy Faupin; David Krejcirik,resolvent,math.AP,math.AP; math-ph; math.SP,2606.26825,https://arxiv.org/abs/2606.26825v1,https://arxiv.org/pdf/2606.26825v1,We derive quantitative uniform resolvent estim...
1,9,2026-06-25T02:34:58+00:00,2026-06-25T02:34:58+00:00,Scattering theory for cavity-assisted spin-mot...,Seigo Kikura; Aruku Senoo; Akihisa Goban; Shin...,photon scattering,quant-ph,quant-ph; physics.atom-ph,2606.26542,https://arxiv.org/abs/2606.26542v1,https://arxiv.org/pdf/2606.26542v1,Cavity-assisted photon scattering (CAPS) is a ...
2,6,2026-06-24T09:39:40+00:00,2026-06-24T09:39:40+00:00,Anomalous topological superradiant phases,You-Qi Lu; Yu-Yu Zhang; Zi-Xiang Hu,topological quantum optics,quant-ph,quant-ph,2606.25635,https://arxiv.org/abs/2606.25635v1,https://arxiv.org/pdf/2606.25635v1,We present a novel set of light-matter topolog...
3,5,2026-06-27T15:04:16+00:00,2026-06-27T15:04:16+00:00,Noncommutative Anisotropic Diffusion in Hilber...,E. Yu. Shchetinin; A. A. Shevchuk; S. I. Salpa...,resolvent,math.AP,math.AP; math-ph,2606.28964,https://arxiv.org/abs/2606.28964v1,https://arxiv.org/pdf/2606.28964v1,This first part of the series builds the analy...
4,5,2026-06-26T18:00:03+00:00,2026-06-26T18:00:03+00:00,Monotonic Impurity Entropy beyond Unitarity: t...,Pradip Kattel; Abay Zhakenov; Natan Andrei,Bethe ansatz,cond-mat.str-el,cond-mat.str-el; cond-mat.mes-hall; hep-th; ma...,2606.28495,https://arxiv.org/abs/2606.28495v1,https://arxiv.org/pdf/2606.28495v1,Quantum impurity models provide a paradigmatic...
5,3,2026-06-24T13:50:09+00:00,2026-06-24T13:50:09+00:00,Brillouin Light Scattering Spectroscopy of Pro...,David Schmoll; Nikolai Kuznetsov; Phillip Rehb...,spin waves,cond-mat.other,cond-mat.other; physics.ins-det; physics.optics,2606.25834,https://arxiv.org/abs/2606.25834v1,https://arxiv.org/pdf/2606.25834v1,Coupling light to magnetic excitations in the ...


In [6]:
radar.dataframe[["score", "published", "title", "matched_keywords", "abstract_url"]].head(10)

,score,published,title,matched_keywords,abstract_url
0,13,2026-06-25T10:05:27+00:00,Quantitative uniform resolvent estimates,resolvent,https://arxiv.org/abs/2606.26825v1
1,9,2026-06-25T02:34:58+00:00,Scattering theory for cavity-assisted spin-mot...,photon scattering,https://arxiv.org/abs/2606.26542v1
2,6,2026-06-24T09:39:40+00:00,Anomalous topological superradiant phases,topological quantum optics,https://arxiv.org/abs/2606.25635v1
3,5,2026-06-27T15:04:16+00:00,Noncommutative Anisotropic Diffusion in Hilber...,resolvent,https://arxiv.org/abs/2606.28964v1
4,5,2026-06-26T18:00:03+00:00,Monotonic Impurity Entropy beyond Unitarity: t...,Bethe ansatz,https://arxiv.org/abs/2606.28495v1
5,3,2026-06-24T13:50:09+00:00,Brillouin Light Scattering Spectroscopy of Pro...,spin waves,https://arxiv.org/abs/2606.25834v1


Search within the returned results:

In [7]:
topic = "waveguide"

mask = (
    radar.dataframe["title"].str.contains(topic, case=False, na=False)
    | radar.dataframe["summary"].str.contains(topic, case=False, na=False)
    | radar.dataframe["matched_keywords"].str.contains(topic, case=False, na=False)
)

radar.dataframe.loc[mask, ["score", "published", "title", "abstract_url"]]

,score,published,title,abstract_url


## Inspect one paper object

The table is convenient, but each paper is also a typed `Paper` model with all fields preserved.

In [8]:
paper = radar.papers[0] if radar.papers else None
paper.model_dump() if paper else "No matching papers found."

{'title': 'Quantitative uniform resolvent estimates',
 'authors': ["Piero D'Ancona", 'Jérémy Faupin', 'David Krejcirik'],
 'published': datetime.datetime(2026, 6, 25, 10, 5, 27, tzinfo=datetime.timezone.utc),
 'updated': datetime.datetime(2026, 6, 25, 10, 5, 27, tzinfo=datetime.timezone.utc),
 'arxiv_id': '2606.26825',
 'abstract_url': 'https://arxiv.org/abs/2606.26825v1',
 'pdf_url': 'https://arxiv.org/pdf/2606.26825v1',
 'primary_category': 'math.AP',
 'categories': ['math.AP', 'math-ph', 'math.SP'],
 'summary': 'We derive quantitative uniform resolvent estimates for Schrödinger operators on the half-line with inverse-square potentials, which provide a sharp behaviour in the limit of large coupling. Our approach is based on a matrix representation of the boundary value of a weighted resolvent. The partial wave decomposition then turns these one-dimensional channel estimates into explicit weighted resolvent estimates for the Laplacian, its inverse-square potential perturbations and fo

## Write reports from the notebook

Use this when you want Markdown and CSV artifacts after reviewing results interactively.

In [9]:
markdown_path, csv_path = radar.write_reports(
    output_dir=PROJECT_ROOT / "reports",
    days=14,
)

markdown_path, csv_path

(PosixPath('/Users/wangyangming/research_project/arxiv-radar/reports/arxiv-radar-2026-07-01.md'),
 PosixPath('/Users/wangyangming/research_project/arxiv-radar/reports/arxiv-radar-2026-07-01.csv'))

## Try temporary keyword or category changes

For persistent changes, edit `arxiv-radar.yaml`. For quick experiments, pass category overrides or temporarily create a modified config object in memory.

In [10]:
optics_only = run_radar(
    config_path=CONFIG_PATH,
    days=14,
    max_results=25,
    categories=["quant-ph", "physics.optics"],
)

optics_only.dataframe[["score", "published", "title", "matched_keywords"]].head(10)

,score,published,title,matched_keywords


## Development rule

Use notebooks for research interaction and quick experiments. If a behavior should become part of the tool, move it into `src/arxiv_radar/` and add a focused test under `tests/`.

Use `uv` for environment and validation commands:

```bash
uv sync
uv run pytest
```